# Sujet 07 — Santander Transaction · Séance 2 : Mesurer honnêtement

Séance : 2
Groupe : [à compléter] (groupe de 3)

| Porteur # | Nom | Étapes assignées |
|---|---|---|
| P1 | [à compléter] | 2.1 Séparer, 2.4 Vérifier que rien n'a fuité, 2.7 Enregistrer l'essai |
| P2 | Prosper [ajouter le nom de famille] | 2.2 Mesurer le bruit, 2.5 Faire tourner un seul modèle |
| P3 | [à compléter] | 2.3 Mesurer le plancher, 2.6 Comparer |

**Déclaration d'usage de l'IA.** Le code de ce carnet a été écrit avec l'aide de Claude (Anthropic), puis exécuté et relu par l'étudiant.

Ce carnet couvre les étapes de **P2** : 2.2 et 2.5. Le mot « contrôle » désigne la partie des données mise de côté pour mesurer (on l'appelait « validation » à la séance 1).

## Préparation
Imports, paramètres, chargement, et deux petites fonctions : le découpage et le modèle. Le découpage ci-dessous est **provisoire** : l'étape 2.1 est celle de P1, il faudra le remplacer par le découpage retenu par l'équipe.

In [ ]:
import os

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = "."   # dossier qui contient train.csv
SEED = 42        # tirage de référence (celui de la séance 1)
N_TIRAGES = 10   # nombre de fois où l'on refait la mesure
VARS = [f"var_{i}" for i in range(200)]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))


def decouper(graine):
    """Découpage provisoire (2.1, à confirmer avec P1) : 75 % apprentissage, 25 % contrôle,
    avec la même proportion de 1 dans les deux parties."""
    return train_test_split(train, test_size=0.25, stratify=train["target"], random_state=graine)


def modele():
    """Un seul modèle, sans réglage : mise à l'échelle des variables + régression logistique.
    max_iter=1000 sert seulement à laisser le calcul finir ; ce n'est pas un réglage de performance."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

## Étape 2.2 — Mesurer le bruit (P2)
**Je fais :** je refais la même mesure plusieurs fois en changeant **seulement le tirage** (la graine du découpage), et je regarde de combien le score bouge.
**Je produis :** la dispersion obtenue, exprimée dans l'unité de la métrique (des points d'AUC).

Ce calcul peut prendre une à trois minutes : le modèle est entraîné 10 fois.

In [ ]:
scores = []
for graine in range(N_TIRAGES):
    tr, va = decouper(graine)
    m = modele().fit(tr[VARS], tr["target"])
    auc = roc_auc_score(va["target"], m.predict_proba(va[VARS])[:, 1])
    scores.append(auc)
    print("tirage", graine, ": AUC =", round(auc, 4))

scores = pd.Series(scores)
print()
print("Moyenne    :", round(scores.mean(), 4))
print("Écart-type :", round(scores.std(), 4))
print("Minimum    :", round(scores.min(), 4), "| Maximum :", round(scores.max(), 4))
print("Étendue (max - min) :", round(scores.max() - scores.min(), 4), "point d'AUC")

**Résultat mesuré :** [à compléter après exécution : moyenne, écart-type et étendue en points d'AUC].
**Lecture :** un écart entre deux modèles plus petit que cette dispersion peut venir du seul hasard du tirage.

## Étape 2.5 — Faire tourner un seul modèle (P2)
**Je fais :** un seul modèle, sans réglage. Je note les variables que je lui donne.
**Je produis :** le modèle, la liste de ses variables, et son score sur le contrôle.

Le `StandardScaler` est dans le pipeline : il apprend sa moyenne et son écart-type sur la partie apprentissage seulement, puis les applique au contrôle. C'est ce qui évite une fuite (étape 2.4, à vérifier avec P1).

In [ ]:
tr, va = decouper(SEED)
m = modele().fit(tr[VARS], tr["target"])
auc = roc_auc_score(va["target"], m.predict_proba(va[VARS])[:, 1])

print("Modèle : régression logistique avec mise à l'échelle, sans réglage")
print("Variables données :", len(VARS), "(" + VARS[0] + " ... " + VARS[-1] + ")")
print("AUC sur le contrôle :", round(auc, 4))
print("Écart avec une règle sans variable (AUC 0,5) :", round(auc - 0.5, 4))

**Résultat mesuré :** [à compléter après exécution : AUC sur le contrôle]. À la séance 1, le même type de modèle avec les 200 variables avait donné environ 0,859 sur notre découpage.